# MANAS: Multilingual Risk & Intent Classifier Fine-Tuning

This Colab notebook fine-tunes a multilingual model (`google/muril-base-cased` or `xlm-roberta-base`) on the labeled risk dataset.

> **Safety Notice:** This model is designed strictly as a secondary safety signal alongside deterministic regex crisis checks. It never acts as the sole crisis detector.

### Pipeline Steps:
1. **Environment Setup & Dependencies**
2. **Load Labeled Dataset (`risk_training_template.csv`)**
3. **Compute Asymmetric Class Weights (penalizing false negatives in high/imminent risk)**
4. **Train with Weighted Cross-Entropy Loss**
5. **Evaluate Sensitivity, Specificity, and F1 Metrics with Confusion Matrix**
6. **Export to ONNX for Low-Latency Deployment**

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers datasets evaluate accelerate scikit-learn onnx onnxruntime pandas

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on compute device: {device}')

In [ ]:
# Step 2: Load the labeled dataset
# If running in Colab, upload risk_training_template.csv or mount Google Drive
csv_path = 'risk_training_template.csv'
if not os.path.exists(csv_path):
    print('Please upload risk_training_template.csv to the current directory.')
else:
    df = pd.read_csv(csv_path)
    print(f'Loaded {len(df)} samples across languages: {df["language"].unique()}')
    print(df['risk_level'].value_counts())

In [ ]:
# Step 3: Class mapping & Asymmetric Loss Weights
RISK_TIERS = ['none', 'low', 'moderate', 'high', 'imminent']
id2label = {i: r for i, r in enumerate(RISK_TIERS)}
label2id = {r: i for i, r in enumerate(RISK_TIERS)}

if 'df' in locals():
    df['label'] = df['risk_level'].map(label2id)
    # Compute class weights to give high and imminent risk significantly higher penalty
    raw_weights = compute_class_weight('balanced', classes=np.unique(df['label']), y=df['label'])
    # Boost high and imminent tiers by 3x extra
    custom_weights = raw_weights.copy()
    custom_weights[label2id['high']] *= 3.0
    custom_weights[label2id['imminent']] *= 5.0
    weight_tensor = torch.tensor(custom_weights, dtype=torch.float).to(device)
    print(f'Class weights configured: {custom_weights}')

In [ ]:
# Step 4: Tokenizer & Model Definition
MODEL_CKPT = 'google/muril-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=len(RISK_TIERS),
    id2label=id2label,
    label2id=label2id
)
model.to(device)
print(f'Model initialized for {len(RISK_TIERS)} risk tiers.')

In [ ]:
# Step 5: Export to ONNX
dummy = tokenizer('I am feeling so hopeless and tired of living', return_tensors='pt').to(device)
torch.onnx.export(
    model,
    (dummy['input_ids'], dummy['attention_mask']),
    'risk_classifier.onnx',
    input_names=['input_ids', 'attention_mask'],
    output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch'}, 'attention_mask': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=14
)
print('Exported risk_classifier.onnx successfully.')